# Encoder

In [ ]:
import math
import time
from collections import Counter
import heapq

Q = 3  # Q-nary Huffman, you can set Q=2 for binary, Q=3 for ternary, etc.

with open('war_and_peace_cn.txt', 'r', encoding='utf-8') as f:
    content = f.read()
counter = Counter(content)
total = sum(counter.values())

symbols = list(counter.keys())
probs = [counter[s] / total for s in symbols]

class Node:
    def __init__(self, prob, symbol=None, children=None):
        self.prob = prob
        self.symbol = symbol
        self.children = children if children else []
    def __lt__(self, other):
        return self.prob < other.prob

def q_nary_huffman(symbols, probs, Q):
    heap = [Node(p, s) for s, p in zip(symbols, probs)]
    heapq.heapify(heap)
    # Padding to make (n-1) mod (Q-1) == 0
    n = len(heap)
    if Q > 1 and n > 1:
        pad = (Q - 1 - (n - 1) % (Q - 1)) % (Q - 1)
        for _ in range(pad):
            heapq.heappush(heap, Node(0, None))
    # Build tree
    while len(heap) > 1:
        children = [heapq.heappop(heap) for _ in range(Q)]
        node = Node(sum(child.prob for child in children), None, children)
        heapq.heappush(heap, node)
    # Traverse tree
    codes = {}
    def assign_codes(node, code=''):
        if node.symbol is not None:
            codes[node.symbol] = code
        else:
            for i, child in enumerate(node.children):
                assign_codes(child, code + str(i))
    assign_codes(heap[0])
    return codes

codes = q_nary_huffman(symbols, probs, Q=Q)

for s in symbols:
    if s == '\n':
        display_s = '\\n'
    elif s == ' ':
        display_s = "' '"
    else:
        display_s = s
    print(f"symbol: {display_s} code: {codes[s]}")

entropy = -sum(p * math.log2(p) / math.log2(Q) for p in probs)
avg_code_len = sum([len(codes[s]) * probs[i] for i, s in enumerate(symbols)])  # Q-nary code length in bits
efficiency = entropy / avg_code_len if avg_code_len > 0 else 0

print(f"\ninfo source entropy: {entropy:.4f} bits")
print(f"avg. codeword length: {avg_code_len:.4f} bits/symbol")
print(f"encoding efficiency: {efficiency * 100:.2f}%")

# Decoder

In [ ]:
# encode the whole text
encoded_text = ''.join([codes[c] for c in content])

# Build decoding map
decode_map = {v: k for k, v in codes.items()}

# Decoder
def q_nary_huffman_decode(encoded_str, decode_map):
    decoded = []
    buffer = ''
    max_code_len = max(len(code) for code in decode_map)
    i = 0
    while i < len(encoded_str):
        buffer = ''
        for l in range(1, max_code_len + 1):
            if i + l > len(encoded_str):
                break
            buffer = encoded_str[i:i+l]
            if buffer in decode_map:
                decoded.append(decode_map[buffer])
                i += l
                break
        else:
            break
    return ''.join(decoded)

# decode and time
start_time = time.time()
decoded_text = q_nary_huffman_decode(encoded_text, decode_map)
end_time = time.time()

print(f"\nwhether decoding is correct: {decoded_text == content}")
print(f"decoding time : {end_time - start_time:.6f} seconds")

# README

This notebook implements Q-nary Huffman coding for text compression and decompression. The value of Q can be set (e.g., Q=2 for binary, Q=3 for ternary). All symbols, including punctuation and whitespace, are encoded and decoded.

---

## Encoder

The encoder performs the following steps:

1. **Read and Count Symbols**  
   The text file is read, and the frequency of each symbol (including characters, spaces, and punctuation) is counted using `collections.Counter`:
   ```python
   with open('war_and_peace.txt', 'r', encoding='utf-8') as f:
       content = f.read()
   counter = Counter(content)
   total = sum(counter.values())
   ```

2. **Calculate Probabilities**  
   The probability of each symbol is calculated:
   ```python
   symbols = list(counter.keys())
   probs = [counter[s] / total for s in symbols]
   ```

3. **Generate Q-nary Huffman Codes**  
   The `q_nary_huffman` function builds a Q-nary Huffman tree and assigns codes to each symbol:
   ```python
   def q_nary_huffman(symbols, probs, Q=2):
       heap = [Node(p, s) for s, p in zip(symbols, probs)]
       heapq.heapify(heap)
       n = len(heap)
       if Q > 1 and n > 1:
           pad = (Q - 1 - (n - 1) % (Q - 1)) % (Q - 1)
           for _ in range(pad):
               heapq.heappush(heap, Node(0, None))
       while len(heap) > 1:
           children = [heapq.heappop(heap) for _ in range(Q)]
           node = Node(sum(child.prob for child in children), None, children)
           heapq.heappush(heap, node)
       codes = {}
       def assign_codes(node, code=''):
           if node.symbol is not None:
               codes[node.symbol] = code
           else:
               for i, child in enumerate(node.children):
                   assign_codes(child, code + str(i))
       assign_codes(heap[0])
       return codes
   codes = q_nary_huffman(symbols, probs, Q=Q)
   ```

4. **Display Codes and Statistics**  
   The code for each symbol is printed, and the source entropy, average codeword length (in bits), and encoding efficiency are calculated:
   ```python
   entropy = -sum(p * math.log2(p) / math.log2(Q) for p in probs)
   avg_code_len = sum([len(codes[s]) * probs[i] for i, s in enumerate(symbols)])
   efficiency = entropy / avg_code_len if avg_code_len > 0 else 0
   print(f"\ninfo source entropy: {entropy:.4f} bits")
   print(f"avg. codeword length: {avg_code_len:.4f} bits/symbol")
   print(f"encoding efficiency: {efficiency * 100:.2f}%")
   ```

---

## Decoder

The decoder performs the following steps:

1. **Encode the Entire Text**  
   The original text is encoded into a Q-nary string using the generated codes:
   ```python
   encoded_text = ''.join([codes[c] for c in content])
   ```

2. **Build Decoding Map**  
   A reverse mapping from codewords to symbols is created:
   ```python
   decode_map = {v: k for k, v in codes.items()}
   ```

3. **Decode the Q-nary String**  
   The `q_nary_huffman_decode` function scans the encoded string and matches codewords to symbols:
   ```python
   def q_nary_huffman_decode(encoded_str, decode_map):
       decoded = []
       buffer = ''
       max_code_len = max(len(code) for code in decode_map)
       i = 0
       while i < len(encoded_str):
           buffer = ''
           for l in range(1, max_code_len + 1):
               if i + l > len(encoded_str):
                   break
               buffer = encoded_str[i:i+l]
               if buffer in decode_map:
                   decoded.append(decode_map[buffer])
                   i += l
                   break
           else:
               break
       return ''.join(decoded)
   ```

4. **Verify and Output**  
   The decoded text is compared with the original to verify correctness, and the decoding time is measured:
   ```python
   start_time = time.time()
   decoded_text = q_nary_huffman_decode(encoded_text, decode_map)
   end_time = time.time()
   print(f"\nwhether decoding is correct: {decoded_text == content}")
   print(f"decoding time : {end_time - start_time:.6f} seconds")
   ```

---

**Note:**  
- All symbols, including punctuation and whitespace, are encoded and decoded.
- The value of Q can be set to use binary, ternary, or higher-order Huffman coding.